In [1]:
import json
import time
from selenium import webdriver
from selenium.webdriver.common.by import By


# ==============================================================================
# 1. CRAWL CHỈ THÔNG SỐ KỸ THUẬT
# ==============================================================================

def crawl_only_specifications(driver, product):
    url_product = "https://fptshop.com.vn" + product["product_url"]

    # Khởi tạo Schema đích ngay từ đầu cho gọn
    product_json = {
        "brand_name": product.get("brand_name"),
        "series_name": None,  # Sẽ lấy từ Breadcrumb nếu có
        "category_name": "Điện thoại",
        "product_info": {
            "name": product.get("product_name"),
            "base_name": product.get("product_name"),
            "short_description": None,
            "detail_description": None,
            "thumbnail_url": product.get("thumbnail_url"),
            "sale": None,
            "warranty_months": 12,
            "status": "ACTIVE",
            "is_featured": False
        },
        "variants": [],  # Bỏ qua không cào biến thể
        "specifications": []
    }

    try:
        driver.get(url_product)
        time.sleep(2.5)  # Chờ trang tải danh mục ban đầu

        # ----------------------------------------------------------
        # LẤY SERIES (TỪ BREADCRUMB)
        # ----------------------------------------------------------
        try:
            breadcrumb = driver.find_element(By.CSS_SELECTOR, "nav.Breadcrumb")
            product_json["series_name"] = (
                breadcrumb.text.split("\n")[-1]
                if "\n" in breadcrumb.text
                else None
            )
        except:
            pass

        # ----------------------------------------------------------
        # CÀO BẢNG THÔNG SỐ PHẲNG
        # ----------------------------------------------------------
        try:
            # Tìm và click nút Xem tất cả thông số
            spec_button = driver.find_element(By.XPATH, "//button[contains(., 'Xem tất cả thông số')]")
            driver.execute_script("arguments[0].click();", spec_button)
            time.sleep(1.5)

            # Quét các khối danh mục (Màn hình, Camera, Pin,...)
            spec_blocks = driver.find_elements(
                By.XPATH,
                "//div[contains(@class,'tab-content') and starts-with(@id,'spec-item-')]"
            )

            specs_flat_list = []
            sort_order = 1

            for block in spec_blocks:
                # Trích xuất tên nhóm danh mục lớn
                try:
                    category_name = block.find_element(
                        By.XPATH,
                        "./div[contains(@class,'b2-semibold') or contains(@class,'text-textOnWhitePrimary')]"
                    ).text.strip()
                except:
                    try:
                        category_name = block.find_element(By.XPATH, "./div[1]").text.strip()
                    except:
                        continue

                # Quét từng hàng thông số con bên trong khối
                rows = block.find_elements(By.XPATH, ".//div[contains(@class,'border-dashed') or contains(@class,'flex-row')]")
                for row in rows:
                    try:
                        key = row.find_element(By.XPATH, "./div[1]").text.strip()
                        value = row.find_element(By.XPATH, "./div[2] | ./*[2]").text.strip().replace("\n", ", ")

                        if not key and not value:
                            continue

                        # Đẩy trực tiếp thành cấu trúc phẳng map với Database của bạn
                        specs_flat_list.append({
                            "spec_category": category_name,
                            "spec_key": key,
                            "spec_value": value,
                            "sort_order": sort_order
                        })
                        sort_order += 1
                    except:
                        continue

            product_json["specifications"] = specs_flat_list
        except Exception as e:
            print(f"  [Lỗi UI] Không mở được bảng thông số của: {product.get('product_name')}")

        return product_json

    except Exception as e:
        print(f"Lỗi kết nối khi crawl {url_product}:", e)
        return None


# ==============================================================================
# 2. BỘ ĐIỀU KHIỂN CHẠY PIPELINE
# ==============================================================================

def main():
    try:
        with open("products.json", "r", encoding="utf-8") as f:
            products = json.load(f)
    except FileNotFoundError:
        print("Lỗi: Không tìm thấy file nguồn 'products.json'.")
        return

    driver = webdriver.Chrome()
    final_products = []

    print("=== BẮT ĐẦU CÀO RIÊNG THÔNG SỐ KỸ THUẬT ===")
    try:
        for index, product in enumerate(products, start=1):
            print(f"[{index}/{len(products)}] Đang cào: {product.get('product_name')}")

            # Gọi hàm cào trực tiếp trả về định dạng đích (Không cần qua hàm Transform trung gian nữa)
            result = crawl_only_specifications(driver, product)
            
            if result:
                final_products.append(result)

            # Auto-save lưu tiến trình liên tục đề phòng rớt mạng giữa chừng
            if index % 10 == 0:
                with open("products_final.json", "w", encoding="utf-8") as f:
                    json.dump(final_products, f, ensure_ascii=False, indent=2)
                print("  [Checkpoint]: Đã lưu dữ liệu dự phòng.")

        # Lưu file thành phẩm cuối cùng
        with open("products_final.json", "w", encoding="utf-8") as f:
            json.dump(final_products, f, ensure_ascii=False, indent=2)

        print(f"\n=== HOÀN THÀNH: Đã xuất thành công {len(final_products)} sản phẩm chỉ chứa thông số! ===")

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

=== BẮT ĐẦU CÀO RIÊNG THÔNG SỐ KỸ THUẬT ===
[1/154] Đang cào: Xiaomi 17T 5G 12GB 512GB
  [Lỗi UI] Không mở được bảng thông số của: Xiaomi 17T 5G 12GB 512GB
[2/154] Đang cào: Xiaomi 17T Pro 5G 12GB 512GB
  [Lỗi UI] Không mở được bảng thông số của: Xiaomi 17T Pro 5G 12GB 512GB
[3/154] Đang cào: Samsung Galaxy S26 Ultra 5G 12GB 256GB
[4/154] Đang cào: Oppo Find X9 Ultra 5G 12GB 512GB
[5/154] Đang cào: Oppo Find X9s 5G 12GB 256GB
[6/154] Đang cào: iPhone 17 Pro Max 256GB
[7/154] Đang cào: Samsung Galaxy Z Fold7 5G 12GB 256GB
[8/154] Đang cào: OPPO Reno15 F 5G 8GB 256GB
[9/154] Đang cào: Xiaomi Redmi Note 15 6GB 128GB
[10/154] Đang cào: iPhone 17 256GB
  [Checkpoint]: Đã lưu dữ liệu dự phòng.
[11/154] Đang cào: Honor X9d 5G 8GB 256GB
[12/154] Đang cào: Samsung Galaxy S25 Ultra 5G 12GB 256GB
[13/154] Đang cào: REDMAGIC 11 Pro 5G 12GB 256GB
[14/154] Đang cào: Xiaomi Poco X7 5G 12GB 512GB
[15/154] Đang cào: Samsung Galaxy A17 8GB 128GB
[16/154] Đang cào: Honor X7d 8GB 128GB
[17/154] Đang cào: 

In [2]:
import json

with open("products_final.json", "r", encoding="utf-8") as f:
    products = json.load(f)

count = sum(
    1
    for product in products
    if product.get("specifications")
)

print(f"Số sản phẩm có specifications: {count}")
print(f"Tổng số sản phẩm: {len(products)}")

Số sản phẩm có specifications: 93
Tổng số sản phẩm: 154
